# Fine-tune the 8B student, then serve it with vLLM

**Phase two, step 2.** Reads the `(image, html)` labels from `teacher-label-tables.ipynb` and LoRA-fine-tunes the 8B student on them with Unsloth, then hot-loads the adapter into vLLM.

**Base model = `Qwen/Qwen3-VL-8B-Instruct`.** Your `qwenvl-8b-instruct`. This must be the *exact* checkpoint vLLM already serves — a LoRA adapter only loads on top of the base it was trained against. If your served base differs, change `STUDENT_MODEL` below to match it and retrain.

**Runs on the GPU box** (the RTX 5000), not this Mac. Two things to know going in:
- `src/model/inference.py` and `src/train/lora.py` have never executed on a real GPU. Budget **30–60 min** for signature mismatches on the first run — that is expected, not breakage.
- Confirm which RTX 5000 this is. RTX 5000 Ada = 32 GB + bf16 (8B trains comfortably in 4-bit); Quadro RTX 5000 = 16 GB Turing, no bf16 (tighten `max_pixels` / `max_seq_length` or drop to a 4B base).

## Setup

GPU box only. Installs Unsloth + torch; restart the kernel afterwards on a fresh runtime.

In [ ]:
# On the GPU box only.
# !pip install -q -r requirements-base.txt
# !pip install -q -r requirements-gpu.txt
# Restart the kernel here before continuing.

In [ ]:
import sys, json
from pathlib import Path
from types import SimpleNamespace

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.train.lora import train, TrainConfig, build_dataset
from src.model.inference import gpu_report
print(gpu_report())

## Config

The prompt rule that matters most (from `CLAUDE.md`): **the instruction used to train must equal the instruction used at inference.** We define `STUDENT_INSTRUCTION` once here and pass it via `instruction=` to both training and serving. Editing `src/model/prompts.py` instead would do nothing — the module is already imported.

Note the student prompt is **concise and has no chain-of-thought.** The teacher reasoned its way to the label offline; the student must emit HTML directly so serving is fast and the output parses cleanly. It keeps the same ignore-list, because at serving it sees the same messy invoices.

`mode='full'` — invoices need the cell *values*, so we train on HTML with text, not the structure-only target the public-data spike used.

In [ ]:
STUDENT_MODEL = 'Qwen/Qwen3-VL-8B-Instruct'   # == the vLLM-served base
MANIFEST      = ROOT / 'data' / 'teacher' / 'labels.jsonl'
OUTPUT_DIR    = ROOT / 'outputs' / 'invoice-lora'
ADAPTER_DIR   = OUTPUT_DIR / 'adapter'

# Concise, no-CoT, same ignore-list as the teacher. This exact string is
# used for training AND for every serving request. Do not diverge them.
STUDENT_INSTRUCTION = (
    'Reconstruct the printed data table in this document as HTML. Preserve '
    'every rowspan, colspan, merged cell, and header hierarchy (use <th> '
    'for headers), keep the natural reading order, and transcribe each '
    'cell\'s printed text. Ignore handwriting, stamps, logos, QR codes and '
    'barcodes, and any text that encodes or labels them. Output raw HTML '
    'starting with <table> and ending with </table>. No markdown fences, '
    'no explanation.'
)
print(STUDENT_INSTRUCTION)

## Load the teacher labels

`build_dataset` / `filter_by_length` in `src/train/lora.py` only read `.image_path` and `.html`, so a `SimpleNamespace` per row is enough — no need to reconstruct full `TableRecord`s with difficulty scores.

In [ ]:
rows = [json.loads(l) for l in MANIFEST.read_text().splitlines() if l.strip()]
records = [SimpleNamespace(uid=r['uid'], image_path=r['image_path'], html=r['html'])
           for r in rows if r['html'].strip().startswith('<table')]
print(f'{len(records)} training tables from {MANIFEST}')
assert records, 'no usable labels -- run teacher-label-tables.ipynb first'

# 20-ish real invoices is tiny. This is a proof the pipeline runs and the
# adapter loads -- not a model that will generalise. Reaching trainable
# volume needs synthetic invoices; see the production plan in memory.
if len(records) < 50:
    print('WARNING: <50 samples. Expect format alignment, not real learning.')

## Train

`TrainConfig.instruction = STUDENT_INSTRUCTION` and `mode='full'` are the two lines that adapt the spike's trainer to production. Everything else (4-bit load, gradient checkpointing, checkpoint-and-resume) is inherited from `src/train/lora.py`.

In [ ]:
cfg = TrainConfig(
    model_id=STUDENT_MODEL,
    output_dir=str(OUTPUT_DIR),
    mode='full',                       # invoices need cell text
    instruction=STUDENT_INSTRUCTION,   # MUST match the serving prompt
    finetune_vision_layers=False,      # language-only -> vLLM can hot-load it
    lora_rank=16, lora_alpha=16,
    epochs=2, learning_rate=2e-4,
    max_seq_length=4096,
)

# Sanity-peek one built example before committing to the run.
sample = build_dataset(records[:1], cfg.mode, cfg.instruction)[0]
print(sample['messages'][0]['content'][1]['text'][:200], '...')
print('target:', sample['messages'][1]['content'][0]['text'][:200], '...')

In [ ]:
model, processor, trainer = train(records, cfg)
print('adapter ->', ADAPTER_DIR)

## Verify the adapter is vLLM-loadable

vLLM reads `adapter_config.json`; its `base_model_name_or_path` must match the served base, and with `finetune_vision_layers=False` the adapter touches language layers only (multimodal vision-tower LoRA is the case vLLM is fussiest about).

In [ ]:
cfgj = json.loads((ADAPTER_DIR / 'adapter_config.json').read_text())
print('base_model_name_or_path:', cfgj.get('base_model_name_or_path'))
print('r / alpha             :', cfgj.get('r'), '/', cfgj.get('lora_alpha'))
print('files                 :', sorted(p.name for p in ADAPTER_DIR.iterdir()))
assert (ADAPTER_DIR / 'adapter_model.safetensors').exists() or \
       (ADAPTER_DIR / 'adapter_model.bin').exists(), 'no adapter weights saved'

## Serve base + adapter with vLLM

Run this in a terminal on the serving box (not a notebook cell — it blocks). `--enable-lora` makes the adapter a hot-loadable module, so the base weights stay shared and you can A/B base-vs-adapter live.

```bash
vllm serve Qwen/Qwen3-VL-8B-Instruct \
  --served-model-name qwen3vl-8b \
  --enable-lora \
  --lora-modules invoice-lora=/abs/path/to/outputs/invoice-lora/adapter \
  --max-lora-rank 16 \
  --max-model-len 8192 \
  --limit-mm-per-prompt image=1 \
  --port 8001
```

`--max-lora-rank` must be ≥ the training `lora_rank` (16). Use a **new port** (8001) so this does not collide with the 35B teacher on 8000.

> Caveat worth a 5-minute check: multimodal-LoRA serving support in vLLM moves fast and is model-specific. If vLLM rejects the vision-model LoRA, the fallback is to **merge** the adapter (`model.save_pretrained_merged`) and serve the merged weights plainly — you lose live A/B but keep the fine-tune.

## Query the served adapter

Same `STUDENT_INSTRUCTION` used in training — that is the whole point. Point `SERVE_URL` at the vLLM instance you just started.

In [ ]:
import base64, mimetypes, requests

SERVE_URL   = 'http://localhost:8001/v1'
ADAPTER_TAG = 'invoice-lora'   # the --lora-modules name
BASE_TAG    = 'qwen3vl-8b'     # the --served-model-name (base)


def data_url(path):
    path = Path(path)
    mime = mimetypes.guess_type(str(path))[0] or 'image/png'
    return f'data:{mime};base64,' + base64.b64encode(path.read_bytes()).decode()


def ask(model_tag, image_path, instruction=STUDENT_INSTRUCTION):
    payload = {
        'model': model_tag, 'temperature': 0.0, 'max_tokens': 4096,
        'messages': [{'role': 'user', 'content': [
            {'type': 'image_url', 'image_url': {'url': data_url(image_path)}},
            {'type': 'text', 'text': instruction},
        ]}],
    }
    r = requests.post(f'{SERVE_URL}/chat/completions',
                      headers={'Authorization': 'Bearer EMPTY'},
                      json=payload, timeout=300)
    r.raise_for_status()
    return r.json()['choices'][0]['message']['content']

test_img = records[0].image_path
print(ask(ADAPTER_TAG, test_img)[:1200])

## Live A/B — base vs adapter

Same image, same prompt, two model tags against one running server. This is the cheapest read on whether the fine-tune changed anything. With ~20 samples expect the visible win to be **format alignment** (attribute order, `<th>` vs `<td>`, whitespace the base gets 'wrong'), not new reasoning — call it that honestly.

In [ ]:
from src.model.prompts import clean_prediction
from src.data.html_utils import extract_cells

for tag in (BASE_TAG, ADAPTER_TAG):
    html = clean_prediction(ask(tag, test_img))
    cells = extract_cells(html)
    spans = sum(c.is_spanning for c in cells)
    print(f'{tag:14s} cells={len(cells):3d}  spanning={spans:3d}  '
          f'starts_with_table={html.startswith("<table")}')

---
**Where this sits in the plan.** This proves the loop end-to-end: teacher labels → LoRA → vLLM-served adapter → live A/B. It does **not** yet prove the fine-tune helps — that needs a human-corrected eval set (the 20 invoices) and trainable volume (synthetic invoices). Data is the bottleneck, not the model.